# 🧠 Notebook 3 — Experimentos (LOSO, Cenários e Níveis)

Parte 3 de 4.

Lê as features por nível (`data/features/<nível>/`) e roda **LOSO** em três cenários, para cada nível de canal e cada modelo (RF, XGBoost). Salva os resultados crus para o Notebook 4 montar gráficos.

### Os dois eixos do estudo
**Eixo generalização** (quem treina/testa):
- **A_intra** — treina e testa no mesmo dataset (LOSO interno).
- **B_cross** — treina em todo dataset X, testa em cada paciente do dataset Y.
- **C_mix** — treina na **união** dos outros datasets, testa em cada paciente do held-out. Comparar **A vs C** responde: *misturar datasets melhora a generalização?*

**Eixo vestível** (quantas regiões): rodar os mesmos cenários em **R5 → R3 → R2 → R1** e medir a queda. No **R1** entra o SeizeIT2 (treina clínicos → testa vestível real).

Em todos, **LOSO** é apenas *como* avaliamos: a unidade deixada de fora é sempre um **paciente**.

## 3.1 Imports, parâmetros e modelos

In [ ]:
import os, re, json, io, warnings, gc, html
import html.parser
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import pywt
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, welch, resample_poly
from scipy.stats  import kurtosis as sp_kurtosis, skew as sp_skew
from tqdm.auto    import tqdm

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
print("✅ Imports OK")
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics         import (accuracy_score, f1_score, recall_score,
                                     precision_score, confusion_matrix)
try:
    from xgboost import XGBClassifier
    XGB_OK = True
except Exception:
    XGB_OK = False
    print("⚠️  xgboost indisponível — será pulado.")

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Diretórios ───────────────────────────────────
ROOT_DIR    = 'data'
CHBMIT_DIR  = os.path.join(ROOT_DIR, 'chb-mit')
SIENA_DIR   = os.path.join(ROOT_DIR, 'siena')
SEIZEIT_DIR = os.path.join(ROOT_DIR, 'seizeit2')
MENDELEY_DIR= os.path.join(ROOT_DIR, 'mendeley')
L1_DIR      = os.path.join(ROOT_DIR, 'level1_signals')
L2_DIR      = os.path.join(ROOT_DIR, 'level2_windows')
FEAT_DIR    = os.path.join(ROOT_DIR, 'features')   # subpastas por nível: features/<level>/
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')
for d in [CHBMIT_DIR, SIENA_DIR, SEIZEIT_DIR, MENDELEY_DIR,
          L1_DIR, L2_DIR, FEAT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Filtragem ──────────────────────────────────
F_HP, F_LP, F_ORDER = 0.5, 40.0, 4
NOTCH = {'CHBMIT': 60.0, 'Siena': 50.0, 'SeizeIT2': 50.0, 'Mendeley': 50.0}

# Reamostragem comum (Siena=512, Mendeley=500, demais=256) → fs único para
# tornar features espectrais comparáveis e permitir juntar datasets.
TARGET_FS = 256

# ── Janelamento ────────────────────────────────
WIN_SEC, OVERLAP = 4, 0.50

# ── Rotulagem ──────────────────────────────────
PRE_SEC, POST_SEC, IGAP_SEC = 10*60, 10*60, 10*60
LBL = dict(interictal=0, preictal=1, ictal=2, postictal=3, unknown=-1)
LBL_NAMES = {v: k for k, v in LBL.items()}

# ── Cap de interictal na extração (controla custo do SeizeIT2) ──────────
MAX_INTER_RATIO = 10   # máx. interictal:eventos por paciente (None desliga)

# ── PACIENTES POR DATASET ──────────────────────────────
PATIENTS = {
    'CHBMIT'  : ['chb01', 'chb02', 'chb03', 'chb04', 'chb05'],
    'Siena'   : ['PN00', 'PN01', 'PN03', 'PN05', 'PN06'],
    'Mendeley': ['p10', 'p11', 'p12', 'p13', 'p14'],
    'SeizeIT2': ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005'],
}
PILOT = {k: v[0] for k, v in PATIENTS.items()}
SEIZEIT_SESSION = 'ses-01'

# ── NÍVEIS DE REDUÇÃO DE CANAIS = subconjuntos de REGIÕES cerebrais ────────
# Por que regiões e não eletrodos fixos? Os datasets usam montagens diferentes
# (CHB-MIT é bipolar, Siena/Mendeley referencial, SeizeIT2 behind-the-ear), então
# "o mesmo eletrodo" não existe entre todos. Mapear cada canal → região e agregar
# por região (1) funciona em qualquer montagem, (2) dá vetor de tamanho fixo para
# poder juntar datasets, e (3) mantém a interpretabilidade (SHAP por região).
REGIONS = ['frontal', 'temporal', 'central', 'parietal', 'occipital']

LEVELS = {
    'R5': ['frontal', 'temporal', 'central', 'parietal', 'occipital'],  # completo (~19 ch)
    'R3': ['frontal', 'temporal', 'central'],                           # reduzido (~8 ch)
    'R2': ['frontal', 'temporal'],                                      # vestível (~4 ch)
    'R1': ['temporal'],                                                 # ultra-compacto / SeizeIT2
}
# Quais datasets participam de cada nível. SeizeIT2 (behind-the-ear, ~região
# temporal) só entra no nível mínimo R1, que é onde ele faz sentido físico.
LEVEL_DATASETS = {
    'R5': ['CHBMIT', 'Siena', 'Mendeley'],
    'R3': ['CHBMIT', 'Siena', 'Mendeley'],
    'R2': ['CHBMIT', 'Siena', 'Mendeley'],
    'R1': ['CHBMIT', 'Siena', 'Mendeley', 'SeizeIT2'],
}

# ── Mapa eletrodo → região (sistema 10-20; aceita variantes T7/T3 etc.) ────
ELECTRODE_REGION = {}
for _e in ['fp1','fp2','f7','f3','fz','f4','f8','af3','af4']:           ELECTRODE_REGION[_e]='frontal'
for _e in ['t3','t4','t5','t6','t7','t8','p7','p8','ft9','ft10','tp7','tp8']: ELECTRODE_REGION[_e]='temporal'
for _e in ['c3','cz','c4','fc1','fc2','cp1','cp2']:                     ELECTRODE_REGION[_e]='central'
for _e in ['p3','pz','p4','po3','po4']:                                 ELECTRODE_REGION[_e]='parietal'
for _e in ['o1','o2','oz']:                                             ELECTRODE_REGION[_e]='occipital'

print("✅ Configurações OK")
print(f"   fs alvo     : {TARGET_FS} Hz | Janela {WIN_SEC}s overlap {int(OVERLAP*100)}%")
print(f"   Pré/Pós     : {PRE_SEC//60}/{POST_SEC//60} min | cap interictal {MAX_INTER_RATIO}:1")
print(f"   Níveis      : " + " | ".join(f"{k}({len(v)} reg)" for k,v in LEVELS.items()))
for k, v in PATIENTS.items():
    print(f"   {k:9s}: {v}")

In [ ]:
def normalize_electrode(name):
    """Limpa um nome de canal: 'EEG Fp1-REF' -> 'fp1'. Para bipolar 'F7-T3'
    devolve o PRIMEIRO nó ('f7'), que define a região do canal."""
    s = str(name).lower()
    s = s.replace('eeg', ' ').replace('-ref', ' ').replace('-le', ' ').replace('ref', ' ')
    s = s.strip()
    # bipolar 'a-b' -> primeiro nó
    if '-' in s:
        s = s.split('-')[0]
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

def channel_region(name, dataset=None):
    """Região cerebral de um canal. SeizeIT2 (behind-the-ear) -> temporal."""
    if dataset == 'SeizeIT2':
        return 'temporal'
    return ELECTRODE_REGION.get(normalize_electrode(name), None)

def regions_present(ch_names, dataset=None):
    """Dict região -> lista de índices de canais naquela região."""
    out = {r: [] for r in REGIONS}
    for i, nm in enumerate(ch_names):
        r = channel_region(nm, dataset)
        if r in out:
            out[r].append(i)
    return out

In [ ]:
CLASSES      = [0, 1, 2, 3]
TARGET_CLASS = LBL['preictal']
RATIO        = 3            # interictal:eventos no treino
K_FEATURES   = 20          # SelectKBest (mutual_info)
VOTE_WINDOW, VOTE_MIN, REFRACTORY_S = 10, 7, 300
PRE_TOL_S    = PRE_SEC
RUN_MODELS   = ['rf', 'xgb'] if XGB_OK else ['rf']   # RF + XGBoost (clássicos)

def apply_ratio_multiclass(X, y, ratio=RATIO):
    ii = np.where(y == LBL['interictal'])[0]; ev = np.where(np.isin(y,[1,2,3]))[0]
    if len(ev)==0: return X[:0], y[:0]
    n_want = min(len(ii), ratio*len(ev))
    sel = ii[::max(1, len(ii)//n_want)][:n_want] if n_want>0 else np.array([],int)
    idx = np.sort(np.concatenate([ev, sel])); return X[idx], y[idx]

def build_model(mk):
    if mk=='xgb': return XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                    random_state=RANDOM_SEED, eval_metric='mlogloss',
                    objective='multi:softprob', num_class=len(CLASSES))
    if mk=='svm': return SVC(kernel='rbf', C=10, gamma='scale',
                    class_weight='balanced', random_state=RANDOM_SEED)
    if mk=='rf':  return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                    random_state=RANDOM_SEED, n_jobs=-1)
    raise ValueError(mk)
print("✅ Parâmetros e modelos definidos | modelos:", RUN_MODELS)

## 3.2 Métricas (por janela e por evento)

Calculadas aqui, por fold, e salvas. O Notebook 4 apenas agrega e plota.

In [ ]:
def per_window_metrics(y_true, y_pred):
    out = {'accuracy': float(accuracy_score(y_true, y_pred))}
    prec = precision_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    rec  = recall_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    f1c  = f1_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    for c in CLASSES:
        n = LBL_NAMES[c]
        out[f'prec_{n}']=float(prec[c]); out[f'rec_{n}']=float(rec[c]); out[f'f1_{n}']=float(f1c[c])
    out['f1_macro'] = float(f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0))
    out['confusion'] = confusion_matrix(y_true, y_pred, labels=CLASSES).tolist()
    return out

def sliding_vote(y_pred, target=TARGET_CLASS, window=VOTE_WINDOW, min_votes=VOTE_MIN):
    y = np.asarray(y_pred); n = len(y); is_t = (y==target).astype(int); fired = np.zeros(n,int)
    if n < window: return fired
    cs = np.cumsum(is_t)
    for i in range(window-1, n):
        s = cs[i] - (cs[i-window] if i-window>=0 else 0)
        if s >= min_votes: fired[i]=1
    return fired

def alarms_from_votes(fired, times_s, refractory_s=REFRACTORY_S):
    al=[]; last=-np.inf
    for i,f in enumerate(fired):
        if f==1:
            t=float(times_s[i])
            if t-last>=refractory_s: al.append(t); last=t
    return al

def labels_to_ictal_events(y_true, times_s):
    y=np.asarray(y_true); ev=[]; inq=False; st=0.0
    for i,v in enumerate(y):
        if v==LBL['ictal'] and not inq: st=float(times_s[i]); inq=True
        elif v!=LBL['ictal'] and inq: ev.append((st, float(times_s[i]))); inq=False
    if inq: ev.append((st, float(times_s[-1])))
    return ev

def event_prediction_metrics(y_true, y_pred, times_s, pre_tol_s=PRE_TOL_S):
    times_s=np.asarray(times_s,float)
    total_h=(times_s.max()-times_s.min()+WIN_SEC)/3600.0 if len(times_s) else 0.0
    crises=labels_to_ictal_events(y_true,times_s)
    alarms=alarms_from_votes(sliding_vote(y_pred), times_s)
    used=set(); leads=[]; npred=0
    for (onset,_e) in crises:
        lo,hi=onset-pre_tol_s,onset; bj,bl=None,None
        for j,t in enumerate(alarms):
            if j in used: continue
            if lo<=t<=hi:
                lead=onset-t
                if bl is None or lead<bl: bl,bj=lead,j
        if bj is not None: used.add(bj); leads.append(bl); npred+=1
    nfalse=len(alarms)-len(used)
    return {'n_crises':len(crises),'n_predicted':npred,'n_false_alarms':nfalse,
            'event_sensitivity': npred/len(crises) if crises else 0.0,
            'event_far_per_hour': nfalse/total_h if total_h>0 else 0.0,
            'lead_times_s':[float(x) for x in leads]}
print("✅ Métricas por janela e por evento definidas")

## 3.3 Motor LOSO

`run_loso` treina no conjunto indicado (com undersampling 1:3, normalização e seleção de features — tudo *fit* só no treino) e avalia cada paciente de teste com dupla avaliação (Realista + Balanceada).

In [ ]:
def load_level_index(level):
    """Reconstrói {key: (X,y,t)} a partir de features/<level>/ no disco."""
    d = os.path.join(FEAT_DIR, level); idx = {}
    if not os.path.isdir(d): return idx
    for f in sorted(os.listdir(d)):
        if f.endswith('_X.npy'):
            key = f[:-6]
            idx[key] = (os.path.join(d,f), os.path.join(d,key+'_y.npy'), os.path.join(d,key+'_t.npy'))
    return idx

def _dataset_of(key): return key.split('__',1)[0]

def run_loso(train_keys, test_keys, level_index, model_key, ratio=RATIO, k=K_FEATURES):
    """Treina no conjunto train_keys (subamostrado) e avalia cada paciente de test_keys."""
    Xtr=[]; ytr=[]
    for k_ in train_keys:
        Xtr.append(np.load(level_index[k_][0])); ytr.append(np.load(level_index[k_][1]))
    if not Xtr: return []
    Xtr=np.nan_to_num(np.vstack(Xtr)); ytr=np.concatenate(ytr).astype(int)
    Xtr,ytr=apply_ratio_multiclass(Xtr,ytr,ratio)
    if len(Xtr)==0 or len(np.unique(ytr))<2: return []
    scaler=StandardScaler().fit(Xtr); Xtr_s=scaler.transform(Xtr)
    kf=min(k, Xtr_s.shape[1])
    sel=SelectKBest(mutual_info_classif, k=kf).fit(Xtr_s, ytr)
    model=build_model(model_key); model.fit(sel.transform(Xtr_s), ytr)

    res=[]
    for tk in test_keys:
        Xte=np.nan_to_num(np.load(level_index[tk][0])); yte=np.load(level_index[tk][1]).astype(int)
        tte=np.load(level_index[tk][2])
        if len(Xte)==0: continue
        Xte_sel=sel.transform(scaler.transform(Xte))
        yp=model.predict(Xte_sel)
        mw=per_window_metrics(yte,yp); me=event_prediction_metrics(yte,yp,tte)
        Xb,yb=apply_ratio_multiclass(Xte,yte,ratio)
        mb=per_window_metrics(yb, model.predict(sel.transform(scaler.transform(Xb)))) if len(Xb) else None
        res.append({'test_dataset':_dataset_of(tk),'test_subject':tk,
                    'window':mw,'event':me,'balanceada':mb})
    return res

## 3.4 Cenários × níveis × modelos

Gera todos os jobs e executa. Cada registro guarda nível, cenário, modelo, paciente de teste e as métricas.

In [ ]:
# Cenários (todos avaliados com LOSO = um paciente de teste por vez):
#   A_intra : treina/testa no MESMO dataset
#   B_cross : treina em TODO dataset X → testa cada paciente do dataset Y
#   C_mix   : treina na UNIÃO dos outros datasets → testa cada paciente do held-out
def scenarios_for_level(level, level_index):
    keys = list(level_index)
    by_ds = {}
    for k in keys: by_ds.setdefault(_dataset_of(k), []).append(k)
    datasets = [d for d in LEVEL_DATASETS[level] if by_ds.get(d)]
    jobs = []   # (scenario, train_keys, test_keys, tag)
    # A — intra-dataset LOSO
    for d in datasets:
        ks = by_ds[d]
        if len(ks) < 2:   # precisa de >=2 pacientes p/ LOSO
            continue
        for tk in ks:
            jobs.append(('A_intra', [x for x in ks if x!=tk], [tk], d))
    # B — cross par a par (só datasets clínicos; SeizeIT2 fica fora do B)
    clin = [d for d in datasets if d!='SeizeIT2']
    for dtr in clin:
        for dte in clin:
            if dtr==dte: continue
            jobs.append(('B_cross', by_ds[dtr], by_ds[dte], f'{dtr}->{dte}'))
    # C — mistura: treina nos demais, testa em cada paciente do held-out
    for dte in datasets:
        train = [k for d in datasets if d!=dte for k in by_ds[d]]
        if not train: continue
        for tk in by_ds[dte]:
            jobs.append(('C_mix', train, [tk], f'mix->{dte}'))
    return jobs

# ── Loop principal: níveis × cenários × modelos ───────────────────────
import time
all_records = []
for level in LEVELS:
    lidx = load_level_index(level)
    if not lidx:
        print(f"⚠️  Nível {level}: sem features — rode o Notebook 2."); continue
    jobs = scenarios_for_level(level, lidx)
    print(f"\n{'='*60}\nNÍVEL {level}  ({len(LEVELS[level])} regiões) — {len(jobs)} jobs × {len(RUN_MODELS)} modelos\n{'='*60}")
    for mk in RUN_MODELS:
        t0=time.time()
        for (scen, trk, tek, tag) in jobs:
            for r in run_loso(trk, tek, lidx, mk):
                r.update({'level':level,'scenario':scen,'model':mk,'tag':tag})
                all_records.append(r)
        print(f"  {mk.upper():4s} ok ({time.time()-t0:.0f}s)")

# salva resultados crus para o Notebook 4
def _slim(r):
    rr = dict(r)
    rr['window'] = {k:v for k,v in r['window'].items()}          # inclui 'confusion'
    rr['balanceada'] = r['balanceada']
    return rr
with open(os.path.join(RESULTS_DIR,'results.json'),'w') as f:
    json.dump([_slim(r) for r in all_records], f)
print(f"\n💾 {len(all_records)} registros salvos em data/results/results.json")

---
✅ **Fim do Notebook 3.** Resultados em `data/results/results.json`. Prossiga para o **Notebook 4 — Métricas & Gráficos**.